# Tarea 2 — EDA y Series de Tiempo
## EO4040 Introducción a la Ciencia de Datos para Economía
**Tecnológico de Monterrey — Gobierno y Transformación Pública**

**Valor total: 80 puntos automáticos + 20 puntos presentación = 100 puntos**

**Instrucciones:**
- Ejecuta las celdas en orden.
- Reemplaza `...` con tu código.
- No modifiques las celdas de configuración ni las de tests.
- Preguntas 1–4: Análisis Exploratorio de Datos (sesión 3).
- Preguntas 5–8: Series de Tiempo (sesión 4).
- Al terminar, descarga tu notebook y entrégalo en Canvas.

In [ ]:
# Instalaciones necesarias (ejecuta primero, ~2-3 min en Colab)
import subprocess, sys
for pkg in ["numpy", "pandas", "matplotlib", "scipy", "statsmodels", "wbgapi", "otter-grader==7.0.0"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Instalaciones completas.")

In [ ]:
# Descarga los tests para verificar tus respuestas (ejecuta una sola vez)
import os, pathlib, subprocess, sys

_tests_dir = pathlib.Path("tests")
_tests_dir.mkdir(exist_ok=True)

_base_url = "https://raw.githubusercontent.com/mmoctezuma/IntroCienciaDatosEcon/master/Tareas/Tarea2/tests"
for _q in range(1, 9):
    _dest = _tests_dir / f"q{_q}.py"
    if not _dest.exists():
        subprocess.check_call(["wget", "-q", f"{_base_url}/q{_q}.py", "-O", str(_dest)])

print("Tests disponibles:", sorted(str(p) for p in _tests_dir.glob("*.py")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import acf as sm_acf
import warnings
warnings.filterwarnings("ignore")

import otter
grader = otter.Notebook()

np.random.seed(42)
print("Importaciones listas.")

---
## Parte 1: Análisis Exploratorio de Datos (Sesión 3)

Indicadores del **Banco Mundial (WDI)** para 27 países:
- **19 países de América Latina:** ARG, BOL, BRA, CHL, COL, CRI, CUB, DOM, ECU, GTM, HND, MEX, NIC, PAN, PER, PRY, SLV, URY, VEN
- **8 países de referencia:** USA, DEU, NOR, KOR, ZAF, NGA, IND, CHN

| Código WDI | Columna |
|---|---|
| NY.GDP.PCAP.PP.CD | pib_pc_ppp |
| SI.POV.GINI | gini |
| SL.UEM.TOTL.ZS | desempleo |
| SE.XPD.TOTL.GD.ZS | gasto_educacion |
| SL.TLF.CACT.FE.ZS | participacion_fem |

In [ ]:
# Esta celda NO se modifica — carga los datos (API o fixture local).
import os, pathlib
paises_latam = [
    "ARG","BOL","BRA","CHL","COL","CRI","CUB","DOM",
    "ECU","GTM","HND","MEX","NIC","PAN","PER","PRY",
    "SLV","URY","VEN",
]
paises_ref   = ["USA","DEU","NOR","KOR","ZAF","NGA","IND","CHN"]
todos_paises = paises_latam + paises_ref

_fixture_wb = pathlib.Path("fixture_wb.csv")
_loaded_wb  = False
if _fixture_wb.exists():
    df_wb = pd.read_csv(_fixture_wb)
    _loaded_wb = True
    print("Datos cargados desde fixture local.")
else:
    try:
        import wbgapi as wb
        indicadores = {
            "NY.GDP.PCAP.PP.CD": "pib_pc_ppp",
            "SI.POV.GINI":        "gini",
            "SL.UEM.TOTL.ZS":     "desempleo",
            "SE.XPD.TOTL.GD.ZS":  "gasto_educacion",
            "SL.TLF.CACT.FE.ZS":  "participacion_fem",
        }
        raw_wb = wb.data.DataFrame(
            list(indicadores.keys()),
            economy=todos_paises,
            time=range(2018, 2024),
            mrv=1,
            skipBlanks=False,
            labels=False,
        )
        df_wb = raw_wb.reset_index()
        rename_map = {k: v for k, v in indicadores.items() if k in df_wb.columns}
        df_wb = df_wb.rename(columns=rename_map)
        if "economy" in df_wb.columns:
            df_wb = df_wb.rename(columns={"economy": "pais"})
        df_wb["region"] = df_wb["pais"].apply(
            lambda c: "latam" if c in paises_latam else "referencia"
        )
        for col in indicadores.values():
            if col not in df_wb.columns:
                df_wb[col] = float("nan")
        cols_keep = ["pais","region"] + list(indicadores.values())
        df_wb = df_wb[cols_keep].reset_index(drop=True)
        _loaded_wb = True
        print("Datos descargados desde API.")
    except Exception as _e:
        raise RuntimeError(f"No se pudo cargar df_wb: {_e}")
print("Filas:", len(df_wb), "  Columnas:", list(df_wb.columns))
df_wb.head()

---
### Pregunta 1 (10 pts) — Análisis univariado categórico

Calcula la **tabla de frecuencias** de la columna `region` en `df_wb`.

Guarda:
- `tabla_freq_region`: `pd.Series` con conteos por categoría, ordenado de mayor a menor.
- `moda_region`: categoría con mayor frecuencia (string).

In [ ]:
...

print("Tabla de frecuencias:")
print(tabla_freq_region)
print("Moda:", moda_region)

In [ ]:
grader.check("q1")

---
### Pregunta 2 (10 pts) — Análisis univariado continuo

Para los países de `region == 'latam'`, calcula estadísticas de `pib_pc_ppp`.

Guarda:
- `desc_pib`: `pd.Series` con claves exactas `'media'`, `'mediana'`, `'desv_std'`, `'minimo'`, `'maximo'`.
- `asimetria_pib`: coeficiente de asimetría de Pearson (float, usa `scipy.stats.skew`).

In [ ]:
...

print("Estadisticas PIB pc PPP (latam):")
print(desc_pib.round(2))
print("Asimetria:", round(asimetria_pib, 4))

In [ ]:
grader.check("q2")

---
### Pregunta 3 (10 pts) — Correlación bivariada

Calcula la **matriz de correlación de Pearson** para las 5 variables continuas de `df_wb`.

Guarda:
- `corr_matrix`: `pd.DataFrame` 5×5 con la matriz de correlación.
- `par_mayor_corr`: tupla `(var1, var2)` con el par de mayor correlación en valor absoluto (excluye diagonal), en **orden alfabético**.

In [ ]:
...

print("Matriz de correlacion:")
print(corr_matrix.round(3))
print("Par con mayor correlacion:", par_mayor_corr)

In [ ]:
grader.check("q3")

---
### Pregunta 4 (10 pts) — Estadísticas por grupo

Compara la **mediana del desempleo** entre regiones.

Guarda:
- `mediana_desempleo_grupo`: `pd.Series` con mediana de `desempleo` por `region`, ordenado de mayor a menor.
- `region_mayor_desempleo`: región con mayor mediana (string).
- `diferencia_medianas`: diferencia `latam - referencia` (float).

In [ ]:
...

print("Mediana desempleo por region:")
print(mediana_desempleo_grupo.round(2))
print("Region mayor desempleo:", region_mayor_desempleo)
print("Diferencia (latam - referencia):", round(diferencia_medianas, 2))

In [ ]:
grader.check("q4")

---
## Parte 2: Series de Tiempo (Sesión 4)

Inflación anual del **Banco Mundial** para COL, DOM, HND:
- **Indicador:** `FP.CPI.TOTL.ZG` — Inflación IPC (% anual)
- **Periodo:** 2000–2023
- **Fuente:** wbgapi — no se necesita descargar archivos externos.

In [ ]:
# Esta celda NO se modifica — carga los datos (API o fixture local).
_fixture_inf = pathlib.Path("fixture_inf.csv")
if _fixture_inf.exists():
    df_inf_raw = pd.read_csv(_fixture_inf)
    df_inf_raw["anio"] = df_inf_raw["anio"].astype(int)
    df_inf_raw = df_inf_raw.sort_values(["pais","anio"]).reset_index(drop=True)
    print("Datos inflacion cargados desde fixture local.")
else:
    try:
        import wbgapi as wb
        paises_st = ["COL", "DOM", "HND"]
        raw_inf = wb.data.DataFrame(
            "FP.CPI.TOTL.ZG",
            economy=paises_st,
            time=range(2000, 2024),
            skipBlanks=False,
            labels=False,
        )
        df_inf_raw = (
            raw_inf
            .T
            .reset_index()
            .melt(id_vars="index", var_name="pais", value_name="inflacion")
            .rename(columns={"index": "anio"})
        )
        df_inf_raw["anio"] = df_inf_raw["anio"].astype(str).str.extract(r"(\d{4})").astype(int)
        df_inf_raw = df_inf_raw.dropna(subset=["anio"]).sort_values(["pais","anio"]).reset_index(drop=True)
        print("Datos inflacion descargados desde API.")
    except Exception as _e:
        raise RuntimeError(f"No se pudo cargar df_inf_raw: {_e}")
print("Forma:", df_inf_raw.shape, "  Paises:", df_inf_raw['pais'].unique())
df_inf_raw.head(8)

---
### Pregunta 5 (10 pts) — Preparación de series de tiempo

Crea una serie de tiempo para **Colombia (COL)**.

Guarda:
- `serie_col`: `pd.Series` con inflación anual de COL, índice `pd.PeriodIndex` frecuencia `'Y'`, nombre `'inflacion_COL'`.
- `n_obs_col`: número de observaciones no nulas (int).
- `anio_max_inflacion_col`: año con mayor inflación (int).

In [ ]:
...

print("Serie COL (primeros 5):")
print(serie_col.head())
print("Obs no nulas:", n_obs_col)
print("Anio max inflacion:", anio_max_inflacion_col)

In [ ]:
grader.check("q5")

---
### Pregunta 6 (10 pts) — Descomposición de serie de tiempo

> **Nota técnica:** Los datos son anuales. Usamos `period=2` para extraer tendencia de mediano plazo y residuos. Interpola los NaN antes de descomponer.

Aplica `seasonal_decompose` a `serie_col` con `model='additive'`, `period=2`, `extrapolate_trend='freq'`.

Guarda:
- `descomp_col`: resultado de `seasonal_decompose`.
- `tendencia_col`: `pd.Series` con el componente de tendencia, sin NaN.
- `residuos_col`: `pd.Series` con el componente de residuos, sin NaN.

In [ ]:
...

print("Longitud tendencia:", len(tendencia_col))
print("Longitud residuos: ", len(residuos_col))
print("Tendencia (primeros 5):")
print(tendencia_col.head().round(4))

In [ ]:
grader.check("q6")

---
### Pregunta 7 (10 pts) — Función de Autocorrelación (ACF)

Calcula la ACF de `serie_col` (sin NaN) hasta el **lag 5**.

Guarda:
- `acf_col`: `np.ndarray` con ACF para lags 0–5 (longitud 6). Usa `sm_acf(serie_col_clean, nlags=5, fft=False)` donde `serie_col_clean = serie_col.dropna()`.
- `lag_mayor_acf`: lag (int) con mayor ACF, **excluyendo lag 0**.

In [ ]:
...

print("ACF lags 0-5:")
for i, v in enumerate(acf_col):
    print(f"  lag {i}: {v:.4f}")
print("Lag mayor ACF (excl. lag 0):", lag_mayor_acf)

In [ ]:
grader.check("q7")

---
### Pregunta 8 (10 pts) — Regresión OLS y diagnóstico Durbin-Watson

Ajusta una regresión OLS: Y = inflación COL, X = año (2000–2023). Incluye constante con `sm.add_constant`.

Guarda:
- `modelo_col`: resultado de `sm.OLS(...).fit()`.
- `pendiente_col`: coeficiente de `anio` (float).
- `dw_col`: estadístico Durbin-Watson sobre los residuos (float).
- `hay_autocorr`: `True` si `dw_col < 1.5` (bool).

In [ ]:
...

print("Pendiente (tendencia anual):", round(pendiente_col, 4))
print("Durbin-Watson:", round(dw_col, 4))
print("Autocorrelacion positiva:", hay_autocorr)

In [ ]:
grader.check("q8")

---
## Entrega

Cuando termines, descarga este notebook y súbelo a Canvas:
- **Colab:** Archivo → Descargar → Descargar .ipynb
- **Jupyter local:** File → Download as → Notebook (.ipynb)